# 01 · Sandbox basics

The whole RaitBox surface in one sitting — run code, move files, run commands,
handle errors, go concurrent, and pick a backend.

**Concepts:** a **sandbox** is an isolated Firecracker microVM in your AWS
account; `run_code` uses a **stateful kernel** (variables persist, errors are
*returned* not raised); `commands.run` is the shell; `AsyncSandbox` runs many at
once; **backends** map the same API onto different AWS compute.

**Prerequisites:** AWS credentials + `AWS_DEFAULT_REGION` (default `agentcore-ci` backend).

In [ ]:
%pip install -q raitbox

### Run code — state persists, errors come back as data

In [ ]:
from raitbox import CommandExitException, Sandbox

with Sandbox() as sbx:
    print("session:", sbx.sandbox_id)

    # kernel state carries across calls, like notebook cells
    sbx.run_code("primes = [n for n in range(2, 60) if all(n % d for d in range(2, n))]")
    print("stdout:", sbx.run_code("print(primes)").logs.stdout)
    print("count :", sbx.run_code("len(primes)").text)   # last-expression value

    # errors are returned on .error, not raised — an agent can read and react
    err = sbx.run_code("primes[999]").error
    print("error :", err.name, "-", err.value)

### Files — text, JSON, and raw bytes

In [ ]:
import json

with Sandbox() as sbx:
    sbx.files.write("cfg/settings.json", json.dumps({"workers": 4, "debug": True}))
    workers = sbx.run_code("import json; print(json.load(open('cfg/settings.json'))['workers'])")
    print("workers:", workers.text)

    sbx.files.write("data/blob.bin", bytes(range(8)))
    print("bytes  :", bytes(sbx.files.read("data/blob.bin", format="bytes")))
    print("listing:", [e.path for e in sbx.files.list("cfg")])

### Commands — stream output, inspect exit codes

In [ ]:
with Sandbox() as sbx:
    r = sbx.commands.run("echo $((6 * 7)) && python3 -c 'import sys; print(sys.version_info[:2])'")
    print("exit:", r.exit_code, "|", r.stdout.strip())

    sbx.commands.run("seq 3", on_stdout=lambda c: print("»", c, end=""))  # live stream

    try:
        sbx.commands.run("test -f /does/not/exist")   # non-zero -> raises
    except CommandExitException as e:
        print("missing file, exit", e.exit_code)

### Concurrency — isolated sandboxes in parallel

In [ ]:
import asyncio

from raitbox import AsyncSandbox

MOD = 1_000_000_007


async def factorial_mod(n: int) -> str:
    async with await AsyncSandbox.create() as sbx:
        return (await sbx.run_code(f"import math; print(math.factorial({n}) % {MOD})")).text or ""


print(await asyncio.gather(*(factorial_mod(n) for n in (50, 100, 200))))

### Choosing a backend
Pick with `backend=` on `Sandbox.create()` or `RAITBOX_BACKEND`:

- **`agentcore-ci`** *(default)* — stateful kernel, scientific libs, no inbound network.
- **`agentcore-runtime`** — shell + interactive shell; needs a deployed runtime ARN.
- **`microvm`** — full OS, internet egress, a public endpoint via `get_host`; needs an image ARN.

Anything a backend can't do raises `NotSupportedError` naming the alternative.

### Recap & next
You used the entire core API. From here: `02-agent-feedback-loop` (an LLM drives
a sandbox), `05-eval-harness` (score model code), `12-background-and-snapshots`
(microvm features).